# Phone Usage Analysis (India)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Phuvanenthran-P/data-analytics-projects/blob/main/phone-usage-analysis/phone_usage_analysis.ipynb)

Exploratory data analysis (EDA) of 17,686 mobile users in India: demographics, screen time, data usage, calls, app usage, spending and recharge cost.

**Goals**
1. Load, inspect and validate the dataset
2. Explore usage patterns with static and interactive charts
3. Test for relationships between variables
4. Report findings honestly, including data quality issues

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn, Plotly

**How to use in Colab:** click **File → Save a copy in Drive**, then **Runtime → Run all**. The dataset loads automatically from GitHub, so no upload is needed.

## 1. Import libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style="whitegrid")

## 2. Load the dataset
Uses the local file if present, otherwise downloads it from this GitHub repo.

In [ ]:
FILE = "phone_usage_india.csv"
URL = "https://raw.githubusercontent.com/Phuvanenthran-P/data-analytics-projects/main/phone-usage-analysis/phone_usage_india.csv"

df = pd.read_csv(FILE if os.path.exists(FILE) else URL)
print(f"Loaded {df.shape[0]:,} rows and {df.shape[1]} columns")
df.head()

## 3. Dataset description

| Column | Meaning |
|---|---|
| User ID | Unique identifier for each user |
| Age, Gender, Location | User demographics (10 cities) |
| Phone Brand, OS | Device details |
| Screen Time (hrs/day) | Total daily screen time |
| Data Usage (GB/month) | Monthly mobile data used |
| Calls Duration (mins/day) | Daily call time |
| Number of Apps Installed | Apps on the device |
| Social Media / Streaming / Gaming Time (hrs/day) | Daily time per activity |
| E-commerce Spend (INR/month) | Monthly online shopping spend |
| Monthly Recharge Cost (INR) | Monthly mobile plan cost |
| Primary Use | Main purpose of the phone |

## 4. Inspect and check data quality

In [ ]:
df.info()

In [ ]:
df.describe().T.round(2)

In [ ]:
numeric_cols = df.select_dtypes("number").columns

print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate User IDs:", df["User ID"].duplicated().sum())
print("Negative values:", int((df[numeric_cols] < 0).sum().sum()))
print(f"Age range: {df['Age'].min()} to {df['Age'].max()}")

The dataset is structurally clean: no missing values, duplicates or negative numbers. Next, check whether the values make sense *logically*.

## 5. Logical validation

In [ ]:
# Does each brand match a plausible operating system?
pd.crosstab(df["Phone Brand"], df["OS"])

In [ ]:
apple_android = df[(df["Phone Brand"] == "Apple") & (df["OS"] == "Android")]
print(f"Apple phones recorded as Android: {len(apple_android):,} of {(df['Phone Brand'] == 'Apple').sum():,}")

# Activity time should not exceed total screen time (assuming activities don't overlap)
activity_cols = ["Social Media Time (hrs/day)", "Streaming Time (hrs/day)", "Gaming Time (hrs/day)"]
df["Activity Time Total"] = df[activity_cols].sum(axis=1)
too_high = df["Activity Time Total"] > df["Screen Time (hrs/day)"]
print(f"Users whose social + streaming + gaming time exceeds total screen time: {too_high.sum():,} ({too_high.mean():.1%})")

**Data quality findings**
- Every brand appears on both Android and iOS in roughly equal numbers (e.g. Apple phones recorded as Android), which is not possible in the real world.
- About three in four users have more activity time than total screen time.

These are signs the dataset is **synthetic (randomly generated)**. The analysis below is still a valid exercise in EDA technique, but its results should not be treated as real-world behavior.

In [ ]:
# Helper column for age-group comparisons
df["Age Group"] = pd.cut(
    df["Age"], bins=[14, 24, 34, 44, 54, 60],
    labels=["15-24", "25-34", "35-44", "45-54", "55-60"],
)
sample = df.sample(2000, random_state=42)  # sample to avoid overplotting in scatter charts

## 6. Exploratory data analysis

### 6.1 Age distribution

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df["Age"], bins=20, kde=True)
plt.title("Age Distribution")
plt.xlabel("Age")
plt.ylabel("Number of users")
plt.show()

### 6.2 Gender distribution

In [ ]:
plt.figure(figsize=(6, 6))
df["Gender"].value_counts().plot(kind="pie", autopct="%1.1f%%", startangle=90)
plt.title("Gender Distribution")
plt.ylabel("")
plt.show()

### 6.3 Screen time vs data usage
2,000 randomly sampled users.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=sample, x="Screen Time (hrs/day)", y="Data Usage (GB/month)", hue="Gender", alpha=0.6)
plt.title("Screen Time vs Data Usage")
plt.show()

### 6.4 Calls duration vs apps installed

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=sample, x="Calls Duration (mins/day)", y="Number of Apps Installed", hue="OS", alpha=0.6)
plt.title("Calls Duration vs Number of Apps Installed")
plt.show()

### 6.5 Monthly recharge cost

In [ ]:
plt.figure(figsize=(10, 4))
sns.boxplot(data=df, x="Monthly Recharge Cost (INR)")
plt.title("Monthly Recharge Cost Distribution")
plt.show()

### 6.6 Screen time by age group and city
Bars start at zero so differences are not exaggerated.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

by_age = df.groupby("Age Group", observed=True)["Screen Time (hrs/day)"].mean()
by_age.plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Average Screen Time by Age Group")
axes[0].set_ylabel("Hours per day")
axes[0].tick_params(axis="x", rotation=0)

by_city = df.groupby("Location")["Screen Time (hrs/day)"].mean().sort_values()
by_city.plot(kind="barh", ax=axes[1], color="seagreen")
axes[1].set_title("Average Screen Time by City")
axes[1].set_xlabel("Hours per day")

plt.tight_layout()
plt.show()

print(f"Age group range: {by_age.min():.2f} to {by_age.max():.2f} hrs/day")
print(f"City range: {by_city.min():.2f} ({by_city.idxmin()}) to {by_city.max():.2f} ({by_city.idxmax()}) hrs/day")

### 6.7 Correlation heatmap

In [ ]:
corr = df[numeric_cols].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1)
plt.title("Correlation Heatmap")
plt.show()

pairs = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack()
strongest = pairs.abs().idxmax()
print(f"Strongest correlation: {strongest[0]} vs {strongest[1]} = {pairs[strongest]:.3f}")

## 7. Dashboard

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

sns.histplot(df["Age"], bins=20, kde=True, ax=axes[0, 0])
axes[0, 0].set_title("Age Distribution")

df["Gender"].value_counts().plot(kind="pie", autopct="%1.1f%%", startangle=90, ax=axes[0, 1])
axes[0, 1].set_title("Gender Distribution")
axes[0, 1].set_ylabel("")

sns.scatterplot(data=sample, x="Screen Time (hrs/day)", y="Data Usage (GB/month)", hue="Gender", alpha=0.6, ax=axes[1, 0])
axes[1, 0].set_title("Screen Time vs Data Usage")

sns.scatterplot(data=sample, x="Calls Duration (mins/day)", y="Number of Apps Installed", hue="OS", alpha=0.6, ax=axes[1, 1])
axes[1, 1].set_title("Calls Duration vs Apps Installed")

plt.tight_layout()
plt.show()

## 8. Interactive charts (Plotly)
Hover, zoom and filter by clicking legend entries.

In [ ]:
fig = px.scatter(
    sample, x="Screen Time (hrs/day)", y="Data Usage (GB/month)",
    color="Gender", hover_data=["Age", "Location", "Phone Brand"],
    title="Screen Time vs Data Usage (2,000 sampled users)",
)
fig.show()

In [ ]:
use_summary = df.groupby("Primary Use")[["Screen Time (hrs/day)", "Data Usage (GB/month)"]].mean().round(2).reset_index()

fig = px.bar(
    use_summary, x="Primary Use", y="Screen Time (hrs/day)",
    text="Screen Time (hrs/day)", title="Average Screen Time by Primary Use",
)
fig.update_yaxes(rangemode="tozero")
fig.show()

## 9. Key findings

1. **Typical user:** about 6.5 hours of screen time per day, 25 GB of data per month, 151 minutes of calls per day, 105 apps installed, ₹1,043 monthly recharge and ₹5,076 monthly e-commerce spend.
2. **No meaningful relationships:** all pairwise correlations between numeric variables are near zero (the strongest is −0.02, between apps installed and e-commerce spend). Screen time does not predict data usage, and age does not predict anything.
3. **Groups look identical:** average screen time varies by only about 0.1 hrs/day across age groups and about 0.25 hrs/day across cities. Gender, OS and primary use show the same flat pattern.
4. **Categories are almost perfectly balanced:** each gender is about one third of users, OS is 50/50, and each primary use is about 20%.
5. **Logical inconsistencies:** Apple phones appear on Android, and about 76% of users have activity time that exceeds their total screen time.

## 10. Conclusion

The dataset is clean in structure but shows the hallmarks of randomly generated data: uniform distributions, zero correlations and impossible brand/OS combinations. The honest conclusion is that it **cannot support real-world claims** about Indian mobile users, which is itself a useful lesson: always validate that data makes sense before drawing insights.

**Next steps**
- Repeat the workflow on real telecom or survey data
- Add statistical tests (e.g. ANOVA across groups) to confirm the lack of differences
- Build a Streamlit dashboard with filters